In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression,LinearRegression,Ridge,ElasticNet
from tqdm import tqdm
from sklearn.ensemble import VotingClassifier,VotingRegressor,BaggingRegressor
from sklearn.metrics import log_loss,accuracy_score,precision_score,f1_score,mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer,make_column_selector

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
concrete=pd.read_csv("D:\Machine Learning\Datasets\Cases\Concrete_Strength\Concrete_Data.csv")
concrete

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [4]:
X,y=concrete.drop('Strength',axis=1),concrete['Strength']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25)

In [11]:
scaler=StandardScaler()

lr=LinearRegression()

lr_pipe=Pipeline([("SCL",scaler),("LR",lr)])
lr_pipe.fit(X_train,y_train)

ridge=Ridge()
ridge_pipe=Pipeline([("SCL",scaler),("RIDGE",ridge)])
ridge_pipe.fit(X_train,y_train)

elastic=ElasticNet()
elastic_pipe=Pipeline([("SCL",scaler),("ELASTIC",elastic)])
elastic_pipe.fit(X_train,y_train)

dtr=DecisionTreeRegressor()

n_estimators=[10,25,50,100]
estimators=[lr_pipe,ridge_pipe,elastic_pipe,dtr]





In [13]:
scores=[]
for e in estimators:
    for n in n_estimators:
        bagg=BaggingRegressor(random_state=25,estimator=e,n_estimators=n)
        bagg.fit(X_train,y_train)
        y_pred=bagg.predict(X_test)
        scores.append([e,n,mean_absolute_error(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=('estimators','n_estimators','Score'))
df_scores.sort_values('Score')

,estimators,n_estimators,Score
15,DecisionTreeRegressor(),100,3.755561
14,DecisionTreeRegressor(),50,3.774158
13,DecisionTreeRegressor(),25,3.838482
12,DecisionTreeRegressor(),10,3.891324
0,"(StandardScaler(), LinearRegression())",10,7.675439
4,"(StandardScaler(), Ridge())",10,7.686206
3,"(StandardScaler(), LinearRegression())",100,7.711915
1,"(StandardScaler(), LinearRegression())",25,7.714196
7,"(StandardScaler(), Ridge())",100,7.722424
5,"(StandardScaler(), Ridge())",25,7.724171
